In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio import mask
import geopandas as gpd

PLOT_ID_INDEX = 0

# ---------------------------
# File paths (edit before running)
# ---------------------------
ms_tif_path = 'path/to/MS/YYYY-MM-DD/ms_orthomosaic.tif'
shp_path = 'path/to/RGB/YYYY-MM-DD/plot_grid.shp'
out_csv_path = 'path/to/output/ms_vi.csv'

# ---------------------------
# Open raster
# ---------------------------
src = rasterio.open(ms_tif_path)
print("num bands:", src.count)
print("band descriptions:", src.descriptions)

# ---------------------------
# Open shapefile
# ---------------------------
plots = gpd.read_file(shp_path)
print("Header\n", plots.dtypes)
print(plots.iloc[1, -1])

def extract_by_mask(src, shape_geom):
    """
    Extract the masked image based on the given shape geometry.

    MS band order:
      Band 1 = Red
      Band 2 = Green
      Band 3 = NIR
      Band 4 = RedEdge
      Band 5 = Alpha (ignored)
    """
    mask_img = mask.mask(dataset=src, shapes=[shape_geom], nodata=0, filled=True, crop=True)[0]
    mask_img = mask_img.astype(np.float32)

    red_mean = np.nanmean(mask_img[0, :, :])
    green_mean = np.nanmean(mask_img[1, :, :])
    nir_mean = np.nanmean(mask_img[2, :, :])
    rededge_mean = np.nanmean(mask_img[3, :, :])

    # NDVI
    ndvi = (nir_mean - red_mean) / (nir_mean + red_mean)

    # gNDVI
    gndvi = (nir_mean - green_mean) / (nir_mean + green_mean)

    # RVI_1
    rvi_1 = nir_mean / red_mean

    # GCI
    gci = (nir_mean / green_mean) - 1.0

    # RGVI
    rgvi = red_mean / green_mean

    # DVI
    dvi = nir_mean - red_mean

    # SAVI
    L = 0.5
    savi = ((nir_mean - red_mean) / (nir_mean + red_mean + L)) * (1.0 + L)

    # MSAVI
    msavi = 0.5 * (
        (2.0 * nir_mean) + 1.0
        - np.sqrt(np.square(2.0 * nir_mean + 1.0) - 8.0 * (nir_mean - red_mean))
    )

    # OSAVI
    osavi = (nir_mean - red_mean) / (nir_mean + red_mean + 0.16)

    # RDVI
    rdvi = np.sqrt(np.square(nir_mean - red_mean) / (nir_mean + red_mean))

    # TVI
    tvi = 60.0 * (nir_mean - green_mean) - 100.0 * (red_mean - green_mean)

    # TSAVI
    a, b = 0.96916, 0.084726
    tsavi = (a * (nir_mean - a * red_mean - b)) / (a * nir_mean + red_mean - a * b)

    # PVI
    a, b = 0.96916, 0.084726
    pvi = (nir_mean - a * red_mean - b) / np.sqrt(1 + np.square(a))

    # SAVI_2
    a, b = 0.96916, 0.084726
    savi_2 = nir_mean / (red_mean - (b / a))

    # ATSAVI
    a, b, x = 0.96916, 0.084726, 0.08
    atsavi = (a * (-a * red_mean - b)) / (a * nir_mean + red_mean - a * b + x * (1 + np.square(a)))

    # NDWI
    ndwi = (green_mean - nir_mean) / (green_mean + nir_mean)

    # RVI_2
    rvi_2 = nir_mean / green_mean

    # MCARI
    mcari = (rededge_mean - red_mean - 0.2 * (rededge_mean - green_mean)) * (rededge_mean / red_mean)

    # MCARI_1
    mcari_1 = 1.2 * (2.5 * (nir_mean - red_mean) - 1.3 * (nir_mean - green_mean))

    # MCARI_2
    mcari_2 = (
        1.5 * (2.5 * (nir_mean - red_mean) - 1.3 * (nir_mean - green_mean))
        * (np.square(2.0 * nir_mean + 1))
        - (6.0 * nir_mean - 5.0 * red_mean)
        - 0.5
    )

    # MTVI_1
    mtvi_1 = 1.2 * (1.2 * (nir_mean - green_mean) - 2.5 * (red_mean - green_mean))

    # MTVI_2
    mtvi_2 = (
        1.5 * (1.2 * (nir_mean - green_mean) - 2.5 * (red_mean - green_mean))
        * (np.square(2 * nir_mean + 1))
        - (6.0 * nir_mean - 5.0 * red_mean)
        - 0.5
    )

    # R_MCARI_MTVI_2
    r_mcari_mtvi2 = (
        ((rededge_mean - red_mean - 0.2 * (rededge_mean - green_mean)) * (rededge_mean / red_mean))
        / (
            (1.5 * (1.2 * nir_mean - green_mean) - 2.5 * (red_mean - green_mean))
            * (np.square(2 * nir_mean + 1) - (6.0 * nir_mean - 5.0 * red_mean) - 0.5)
        )
    )

    # DATT
    datt = (nir_mean - rededge_mean) / (nir_mean - red_mean)

    # NDCI
    ndci = (rededge_mean - green_mean) / (rededge_mean + green_mean)

    # PSRI
    psri = (red_mean - green_mean) / rededge_mean

    # PSRI2
    psri2 = (red_mean - green_mean) / nir_mean

    # PSRI3
    psri3 = (rededge_mean - red_mean) / nir_mean

    # SPVI
    spvi = 0.4 * 3.7 * (nir_mean - red_mean) - 1.2 * np.absolute(green_mean - red_mean)

    # TCARI
    tcari = 3.0 * ((rededge_mean - red_mean) - 0.2 * (rededge_mean - green_mean) * (rededge_mean / red_mean))

    # R_TCARI_OSAVI
    r_tcari_osavi = (
        3.0 * ((rededge_mean - red_mean) - 0.2 * (rededge_mean - green_mean) * (rededge_mean / red_mean))
    ) / ((nir_mean - red_mean) / (nir_mean + red_mean + 0.16))

    # RERI
    reri = (rededge_mean - red_mean) / nir_mean

    # NDRE
    ndre = (nir_mean - rededge_mean) / (nir_mean + rededge_mean)

    # MTCI
    mtci = (nir_mean - rededge_mean) / (rededge_mean - red_mean)

    # EVI_2
    evi_2 = 2.5 * ((nir_mean - red_mean) / (nir_mean + 2.4 * red_mean + 1.0))

    # RECI
    reci = (nir_mean / rededge_mean) - 1

    # NGRDI
    ngrdi = (green_mean - red_mean) / (green_mean + red_mean)

    # ARI_2
    ari_2 = nir_mean * ((1.0 / green_mean) - (1.0 / rededge_mean))

    # CRI_2
    cri_2 = (1.0 / green_mean) - (1.0 / rededge_mean)

    # NRVI
    rvi = nir_mean / red_mean
    nrvi = (1 / rvi - 1) / (1 / rvi + 1)

    # OSAVI_2
    osavi_2 = ((1 + 0.16) * (nir_mean - red_mean)) / (nir_mean + red_mean + 0.16)

    # MEVI
    mevi = (2.5 * (nir_mean - red_mean)) / (nir_mean + 2.4 * red_mean + 1)

    # TVI_2
    ndvi_2 = (nir_mean - red_mean) / (nir_mean + red_mean)
    tvi_2 = np.sqrt(ndvi_2 + 0.5)

    # MCARI_OSAVI
    mcari_4 = (rededge_mean - red_mean - 0.2 * (rededge_mean - green_mean)) * (rededge_mean / red_mean)
    osavi_tmp = ((1 + 0.16) * (nir_mean - red_mean)) / (nir_mean + red_mean + 0.16)
    mcari_osavi = mcari_4 / osavi_tmp

    # TCARI_OSAVI
    tcari_4 = 3.0 * ((rededge_mean - red_mean) - 0.2 * (rededge_mean - green_mean) * (rededge_mean / red_mean))
    osavi_tmp2 = ((1 + 0.16) * (nir_mean - red_mean)) / (nir_mean + red_mean + 0.16)
    tcari_osavi = tcari_4 / osavi_tmp2

    # WDRVI
    L = 0.12
    wdrvi = (L * nir_mean - red_mean) / (L * nir_mean + red_mean)

    return (
        green_mean, red_mean, rededge_mean, nir_mean, ndvi, gndvi, rvi_1, gci, rgvi, dvi,
        savi, msavi, osavi, rdvi, tvi, tsavi, pvi, savi_2, atsavi, ndwi, rvi_2, mcari,
        mcari_1, mcari_2, mtvi_1, mtvi_2, r_mcari_mtvi2, datt, ndci, psri, psri2, psri3,
        spvi, tcari, r_tcari_osavi, reri, ndre, mtci, evi_2, reci, ngrdi, ari_2, cri_2,
        nrvi, osavi_2, mevi, tvi_2, mcari_osavi, tcari_osavi, wdrvi
    )

# ---------------------------
# Apply this function to all features
# ---------------------------
num_features = plots.shape[0]

list_plots = []
list_green = []
list_red = []
list_rededge = []
list_nir = []
list_ndvi = []
list_gndvi = []
list_rvi_1 = []
list_gci = []
list_rgvi = []
list_dvi = []
list_savi = []
list_msavi = []
list_osavi = []
list_rdvi = []
list_tvi = []
list_tsavi = []
list_pvi = []
list_savi_2 = []
list_atsavi = []
list_ndwi = []
list_rvi_2 = []
list_mcari = []
list_mcari_1 = []
list_mcari_2 = []
list_mtvi_1 = []
list_mtvi_2 = []
list_r_mcari_mtvi2 = []
list_datt = []
list_ndci = []
list_psri = []
list_psri2 = []
list_psri3 = []
list_spvi = []
list_tcari = []
list_r_tcari_osavi = []
list_reri = []
list_ndre = []
list_mtci = []
list_evi_2 = []
list_reci = []
list_ngrdi = []
list_ari_2 = []
list_cri_2 = []
list_nrvi = []
list_osavi_2 = []
list_mevi = []
list_tvi_2 = []
list_mcari_osavi = []
list_tcari_osavi = []
list_wdrvi = []

for i in range(num_features):
    shape_geom = plots.iloc[i, -1]
    plotid = plots.iloc[i, PLOT_ID_INDEX]

    (
        green_mean, red_mean, rededge_mean, nir_mean, ndvi, gndvi, rvi_1, gci, rgvi, dvi,
        savi, msavi, osavi, rdvi, tvi, tsavi, pvi, savi_2, atsavi, ndwi, rvi_2, mcari,
        mcari_1, mcari_2, mtvi_1, mtvi_2, r_mcari_mtvi2, datt, ndci, psri, psri2, psri3,
        spvi, tcari, r_tcari_osavi, reri, ndre, mtci, evi_2, reci, ngrdi, ari_2, cri_2,
        nrvi, osavi_2, mevi, tvi_2, mcari_osavi, tcari_osavi, wdrvi
    ) = extract_by_mask(src, shape_geom)

    list_plots.append(plotid)
    list_green.append(green_mean)
    list_red.append(red_mean)
    list_rededge.append(rededge_mean)
    list_nir.append(nir_mean)
    list_ndvi.append(ndvi)
    list_gndvi.append(gndvi)
    list_rvi_1.append(rvi_1)
    list_gci.append(gci)
    list_rgvi.append(rgvi)
    list_dvi.append(dvi)
    list_savi.append(savi)
    list_msavi.append(msavi)
    list_osavi.append(osavi)
    list_rdvi.append(rdvi)
    list_tvi.append(tvi)
    list_tsavi.append(tsavi)
    list_pvi.append(pvi)
    list_savi_2.append(savi_2)
    list_atsavi.append(atsavi)
    list_ndwi.append(ndwi)
    list_rvi_2.append(rvi_2)
    list_mcari.append(mcari)
    list_mcari_1.append(mcari_1)
    list_mcari_2.append(mcari_2)
    list_mtvi_1.append(mtvi_1)
    list_mtvi_2.append(mtvi_2)
    list_r_mcari_mtvi2.append(r_mcari_mtvi2)
    list_datt.append(datt)
    list_ndci.append(ndci)
    list_psri.append(psri)
    list_psri2.append(psri2)
    list_psri3.append(psri3)
    list_spvi.append(spvi)
    list_tcari.append(tcari)
    list_r_tcari_osavi.append(r_tcari_osavi)
    list_reri.append(reri)
    list_ndre.append(ndre)
    list_mtci.append(mtci)
    list_evi_2.append(evi_2)
    list_reci.append(reci)
    list_ngrdi.append(ngrdi)
    list_ari_2.append(ari_2)
    list_cri_2.append(cri_2)
    list_nrvi.append(nrvi)
    list_osavi_2.append(osavi_2)
    list_mevi.append(mevi)
    list_tvi_2.append(tvi_2)
    list_mcari_osavi.append(mcari_osavi)
    list_tcari_osavi.append(tcari_osavi)
    list_wdrvi.append(wdrvi)

# ---------------------------
# Create the dataframe
# ---------------------------
mean_indicies = pd.DataFrame({
    'var': list_plots,
    'Mean Green': list_green,
    'Mean Red': list_red,
    'Mean RedEdge': list_rededge,
    'Mean NIR': list_nir,
    'NDVI': list_ndvi,
    'gNDVI': list_gndvi,
    'RVI_1': list_rvi_1,
    'GCI': list_gci,
    'RGVI': list_rgvi,
    'DVI': list_dvi,
    'SAVI': list_savi,
    'MSAVI': list_msavi,
    'OSAVI': list_osavi,
    'RDVI': list_rdvi,
    'TVI': list_tvi,
    'TSAVI': list_tsavi,
    'PVI': list_pvi,
    'SAVI_2': list_savi_2,
    'ATSAVI': list_atsavi,
    'NDWI': list_ndwi,
    'RVI_2': list_rvi_2,
    'MCARI': list_mcari,
    'MCARI_1': list_mcari_1,
    'MCARI_2': list_mcari_2,
    'MTVI_1': list_mtvi_1,
    'MTVI_2': list_mtvi_2,
    'R_MCARI_MTVI_2': list_r_mcari_mtvi2,
    'DATT': list_datt,
    'NDCI': list_ndci,
    'PSRI': list_psri,
    'PSRI2': list_psri2,
    'PSRI3': list_psri3,
    'SPVI': list_spvi,
    'TCARI': list_tcari,
    'R_TCARI_OSAVI': list_r_tcari_osavi,
    'RERI': list_reri,
    'NDRE': list_ndre,
    'MTCI': list_mtci,
    'EVI_2': list_evi_2,
    'RECI': list_reci,
    'NGRDI': list_ngrdi,
    'ARI_2': list_ari_2,
    'CRI_2': list_cri_2,
    'NRVI': list_nrvi,
    'OSAVI_2': list_osavi_2,
    'MEVI': list_mevi,
    'TVI_2': list_tvi_2,
    'MCARI_OSAVI': list_mcari_osavi,
    'TCARI_OSAVI': list_tcari_osavi,
    'WDRVI': list_wdrvi,
})

# Keep one row per polygon
df = pd.concat(
    [plots.reset_index(drop=True), mean_indicies.drop(columns=['var']).reset_index(drop=True)],
    axis=1
)

df.to_csv(out_csv_path, index=None, header=True)

print("Finished. Output written to:")
print(out_csv_path)
print("Output rows:", df.shape[0])

src.close()